# ⚡ HyperHeat — Electric Heat Business Case
### Techno-Economic Analysis: Should industrial plants switch from gas to electric heating?

**Two industries modelled:**
- 🔩 **Steel** — Reheating Furnace (500,000 t/year, 1200°C)
- 🏗️ **Cement** — Rotary Kiln (1,000,000 t/year, 1450°C)

**Key question:** At current 2025 prices, does switching from gas burners to HyperHeat electric heating make financial sense?

---
> **All data sourced from:** Eurostat 2025, IEA Reports, ICE EU ETS market, EEA, IPCC AR6, BNEF

## 🔩 Part 1: Steel Reheating Furnace

In [1]:
# ── STEEL TEA: ASSUMPTIONS ──────────────────────────────────────────

PLANT = {
    "annual_production_tonnes": 500_000,   # World Steel Association
    "heat_demand_gj_per_tonne": 1.2,       # IEA Iron & Steel 2023
    "installed_gas_power_mw":   120,
}

GAS      = {"efficiency": 0.60, "price_eur_per_kwh": 0.045, "co2_kg_per_kwh": 0.202}
ELECTRIC = {"efficiency": 0.95, "price_eur_per_kwh": 0.12,  "co2_kg_per_kwh": 0.095}
CARBON   = {"ets_2025": 70, "ets_2030": 126}
CAPEX    = {"eur_per_kw": 500, "decommission": 200_000, "om_pct": 0.02}
FINANCE  = {"wacc": 0.08, "years": 15, "escalation": 0.02}
GJ_KWH   = 277.78

# ── CORE FUNCTIONS ──────────────────────────────────────────────────

def heat_kwh():        return PLANT["annual_production_tonnes"] * PLANT["heat_demand_gj_per_tonne"] * GJ_KWH
def energy_in(s):      return heat_kwh() / s["efficiency"]
def elec_mw():         return PLANT["installed_gas_power_mw"] * GAS["efficiency"] / ELECTRIC["efficiency"]
def energy_cost(s):    return energy_in(s) * s["price_eur_per_kwh"]
def co2(s):            return energy_in(s) * s["co2_kg_per_kwh"] / 1000
def c_fine(s, ets):    return co2(s) * ets
def om(lbl):           return (elec_mw() if lbl=="e" else PLANT["installed_gas_power_mw"]) * 1000 * CAPEX["eur_per_kw"] * CAPEX["om_pct"]
def total(s, lbl, ets):return energy_cost(s) + c_fine(s,ets) + om(lbl)
def capex():           return elec_mw()*1000*CAPEX["eur_per_kw"] + CAPEX["decommission"]
def saving(ets):       return total(GAS,"g",ets) - total(ELECTRIC,"e",ets)
def payback(ets):      return capex()/saving(ets) if saving(ets)>0 else None
def npv(ets):
    pv = sum(saving(ets)*(1+FINANCE["escalation"])**y/(1+FINANCE["wacc"])**(y+1) for y in range(FINANCE["years"]))
    return pv - capex()
def fmt(n):            return f'€{n/1e6:,.2f}M' if abs(n)>=1e6 else f'€{n:,.0f}'

print('✅ Steel assumptions loaded')

✅ Steel assumptions loaded


In [2]:
# ── STEEL: CURRENT CASE (2025 grid prices) ──────────────────────────

ELECTRIC["price_eur_per_kwh"] = 0.12
ets = CARBON["ets_2025"]

print('═'*60)
print('  STEEL TEA — CURRENT CASE (2025 grid €0.12/kWh, ETS €70/t)')
print('═'*60)
print(f'  {"":<32} {"GAS":>12} {"ELECTRIC":>12}')
print('─'*60)
print(f'  {"Energy needed (GWh/year)":<32} {energy_in(GAS)/1e6:>11.1f} {energy_in(ELECTRIC)/1e6:>11.1f}')
print(f'  {"Energy cost/year":<32} {fmt(energy_cost(GAS)):>12} {fmt(energy_cost(ELECTRIC)):>12}')
print(f'  {"CO₂ emissions (tonnes)":<32} {co2(GAS):>11,.0f} {co2(ELECTRIC):>11,.0f}')
print(f'  {"Carbon fine/year":<32} {fmt(c_fine(GAS,ets)):>12} {fmt(c_fine(ELECTRIC,ets)):>12}')
print(f'  {"O&M/year":<32} {fmt(om("g")):>12} {fmt(om("e")):>12}')
print('─'*60)
print(f'  {"TOTAL ANNUAL COST":<32} {fmt(total(GAS,"g",ets)):>12} {fmt(total(ELECTRIC,"e",ets)):>12}')
print('─'*60)
s = saving(ets)
pb = payback(ets)
n = npv(ets)
print(f'  Annual saving (Gas minus Electric) : {fmt(s)}')
print(f'  Total CAPEX                        : {fmt(capex())}')
print(f'  Payback period                     : {f"{pb:.1f} years" if pb else "❌  Not viable — electric costs MORE"}')
print(f'  NPV (15 years)                     : {fmt(n)}  {"✅ INVEST" if n>0 else "❌ NOT VIABLE"}')
print('═'*60)

════════════════════════════════════════════════════════════
  STEEL TEA — CURRENT CASE (2025 grid €0.12/kWh, ETS €70/t)
════════════════════════════════════════════════════════════
                                          GAS     ELECTRIC
────────────────────────────────────────────────────────────
  Energy needed (GWh/year)               277.8        175.5
  Energy cost/year                     €12.50M      €21.05M
  CO₂ emissions (tonnes)                56,112       16,667
  Carbon fine/year                       €3.93M       €1.17M
  O&M/year                               €1.20M     €757,895
────────────────────────────────────────────────────────────
  TOTAL ANNUAL COST                    €17.63M      €22.98M
────────────────────────────────────────────────────────────
  Annual saving (Gas minus Electric) : €-5.35M
  Total CAPEX                        : €38.09M
  Payback period                     : ❌  Not viable — electric costs MORE
  NPV (15 years)                     : €-89.4

In [3]:
# ── STEEL: BEST CASE (Renewable PPA + ETS 2030 forecast) ────────────

ELECTRIC["price_eur_per_kwh"] = 0.07   # Renewable PPA deal
ets = CARBON["ets_2030"]               # BloombergNEF / GMK Center consensus

print('═'*60)
print('  STEEL TEA — BEST CASE (PPA €0.07/kWh, ETS €126/t 2030)')
print('  Lever 1: Renewable PPA  €0.12 → €0.07/kWh')
print('  Lever 2: ETS carbon price  €70 → €126/tonne by 2030')
print('  Lever 3: Green steel premium (BMW/Volvo already paying €75/t)')
print('═'*60)
print(f'  {"":<32} {"GAS":>12} {"ELECTRIC":>12}')
print('─'*60)
print(f'  {"Energy cost/year":<32} {fmt(energy_cost(GAS)):>12} {fmt(energy_cost(ELECTRIC)):>12}')
print(f'  {"CO₂ emissions (tonnes)":<32} {co2(GAS):>11,.0f} {co2(ELECTRIC):>11,.0f}')
print(f'  {"Carbon fine/year":<32} {fmt(c_fine(GAS,ets)):>12} {fmt(c_fine(ELECTRIC,ets)):>12}')
print(f'  {"O&M/year":<32} {fmt(om("g")):>12} {fmt(om("e")):>12}')
print('─'*60)
print(f'  {"TOTAL ANNUAL COST":<32} {fmt(total(GAS,"g",ets)):>12} {fmt(total(ELECTRIC,"e",ets)):>12}')
print('─'*60)
s = saving(ets)
pb = payback(ets)
n = npv(ets)
print(f'  Annual saving                      : {fmt(s)}')
print(f'  Total CAPEX                        : {fmt(capex())}')
print(f'  Payback period                     : {f"{pb:.1f} years ✅" if pb else "❌"}')
print(f'  NPV (15 years)                     : {fmt(n)}  {"✅ STRONG INVEST" if n>0 else "❌"}')
co2_saved = co2(GAS) - co2(ELECTRIC)
print(f'\n  CO₂ saved/year                     : {co2_saved:,.0f} tonnes')
print(f'  Equivalent cars off road           : {co2_saved/1.2:,.0f} cars/year')
print('═'*60)

════════════════════════════════════════════════════════════
  STEEL TEA — BEST CASE (PPA €0.07/kWh, ETS €126/t 2030)
  Lever 1: Renewable PPA  €0.12 → €0.07/kWh
  Lever 2: ETS carbon price  €70 → €126/tonne by 2030
  Lever 3: Green steel premium (BMW/Volvo already paying €75/t)
════════════════════════════════════════════════════════════
                                          GAS     ELECTRIC
────────────────────────────────────────────────────────────
  Energy cost/year                     €12.50M      €12.28M
  CO₂ emissions (tonnes)                56,112       16,667
  Carbon fine/year                       €7.07M       €2.10M
  O&M/year                               €1.20M     €757,895
────────────────────────────────────────────────────────────
  TOTAL ANNUAL COST                    €20.77M      €15.14M
────────────────────────────────────────────────────────────
  Annual saving                      : €5.63M
  Total CAPEX                        : €38.09M
  Payback period      

## 🏗️ Part 2: Cement Rotary Kiln

In [4]:
# ── CEMENT TEA: ASSUMPTIONS ─────────────────────────────────────────
# KEY DIFFERENCE FROM STEEL:
# 60% of cement CO₂ = calcination (limestone chemistry) → UNAVOIDABLE
# HyperHeat only eliminates the 40% from fuel combustion

CP = {
    "cement_output":     1_000_000,   # Large EU plant
    "clinker_ratio":     0.75,         # Cembureau 2024
    "heat_gj_per_tonne": 3.5,          # IEA Cement 2023
    "gas_mw":            120,
}
CGAS  = {"efficiency": 0.55, "price_eur_per_kwh": 0.045, "co2_kg_per_kwh": 0.202}
CELEC = {"efficiency": 0.90, "price_eur_per_kwh": 0.12,  "co2_kg_per_kwh": 0.095}
CO2S  = {"total": 0.65, "calcination": 0.53, "fuel": 0.12}  # per tonne cement
CCARB = {"ets_2025": 70, "ets_2030": 126}
CCAPEX= {"eur_per_kw": 500, "om_pct": 0.02}
CFIN  = {"wacc": 0.08, "years": 15, "escalation": 0.02}

def c_clinker():       return CP["cement_output"] * CP["clinker_ratio"]
def c_heat():          return c_clinker() * CP["heat_gj_per_tonne"] * GJ_KWH
def c_energy(s):       return c_heat() / s["efficiency"]
def c_elec_mw():       return CP["gas_mw"] * CGAS["efficiency"] / CELEC["efficiency"]
def c_ecost(s):        return c_energy(s) * s["price_eur_per_kwh"]
def c_fco2(s):         return c_energy(s) * s["co2_kg_per_kwh"] / 1000
def c_cfine(s, ets):   return c_fco2(s) * ets
def c_capex():         return c_elec_mw() * 1000 * CCAPEX["eur_per_kw"]
def c_save(ets):       return (c_ecost(CGAS)-c_ecost(CELEC)) + (c_cfine(CGAS,ets)-c_cfine(CELEC,ets))
def c_pb(ets):         return c_capex()/c_save(ets) if c_save(ets)>0 else None
def c_npv(ets):
    pv = sum(c_save(ets)*(1+CFIN["escalation"])**y/(1+CFIN["wacc"])**(y+1) for y in range(CFIN["years"]))
    return pv - c_capex()

print('✅ Cement assumptions loaded')
print(f'   CO₂ split per tonne cement:')
print(f'   Calcination : {CO2S["calcination"]} t/t  ← chemistry, unavoidable without CCS')
print(f'   Fuel        : {CO2S["fuel"]} t/t  ← what HyperHeat eliminates')
print(f'   Total today : {CO2S["total"]} t/t')

✅ Cement assumptions loaded
   CO₂ split per tonne cement:
   Calcination : 0.53 t/t  ← chemistry, unavoidable without CCS
   Fuel        : 0.12 t/t  ← what HyperHeat eliminates
   Total today : 0.65 t/t


In [5]:
# ── CEMENT: CURRENT CASE ────────────────────────────────────────────

CELEC["price_eur_per_kwh"] = 0.12
ets = CCARB["ets_2025"]

print('═'*62)
print('  CEMENT TEA — CURRENT CASE (2025 grid €0.12/kWh, ETS €70/t)')
print('═'*62)
print(f'  {"":<35} {"GAS":>12} {"ELECTRIC":>12}')
print('─'*62)
print(f'  {"Energy cost/year":<35} {fmt(c_ecost(CGAS)):>12} {fmt(c_ecost(CELEC)):>12}')
print(f'  {"Fuel CO₂ (tonnes)":<35} {c_fco2(CGAS):>11,.0f} {c_fco2(CELEC):>11,.0f}')
print(f'  {"Carbon fine on fuel CO₂":<35} {fmt(c_cfine(CGAS,ets)):>12} {fmt(c_cfine(CELEC,ets)):>12}')
print('─'*62)
s = c_save(ets); pb = c_pb(ets); n = c_npv(ets)
print(f'  Annual saving               : {fmt(s)}')
print(f'  CAPEX                       : {fmt(c_capex())}')
print(f'  Payback                     : {f"{pb:.1f} years" if pb else "❌  Not viable"}')
print(f'  NPV (15yr)                  : {fmt(n)}  {"✅" if n>0 else "❌ NOT VIABLE"}')
print('═'*62)

══════════════════════════════════════════════════════════════
  CEMENT TEA — CURRENT CASE (2025 grid €0.12/kWh, ETS €70/t)
══════════════════════════════════════════════════════════════
                                          GAS     ELECTRIC
────────────────────────────────────────────────────────────────
  Energy cost/year                     €59.66M      €97.22M
  Fuel CO₂ (tonnes)                    267,805       76,968
  Carbon fine on fuel CO₂              €18.75M       €5.39M
────────────────────────────────────────────────────────────────
  Annual saving               : €-24.20M
  CAPEX                       : €36.67M
  Payback                     : ❌  Not viable
  NPV (15yr)                  : €-268.92M  ❌ NOT VIABLE
══════════════════════════════════════════════════════════════


In [6]:
# ── CEMENT: BEST CASE ───────────────────────────────────────────────

CELEC["price_eur_per_kwh"] = 0.07
ets = CCARB["ets_2030"]

print('═'*62)
print('  CEMENT TEA — BEST CASE (PPA €0.07/kWh, ETS €126/t 2030)')
print('═'*62)
print(f'  {"":<35} {"GAS":>12} {"ELECTRIC":>12}')
print('─'*62)
print(f'  {"Energy cost/year":<35} {fmt(c_ecost(CGAS)):>12} {fmt(c_ecost(CELEC)):>12}')
print(f'  {"Fuel CO₂ (tonnes)":<35} {c_fco2(CGAS):>11,.0f} {c_fco2(CELEC):>11,.0f}')
print(f'  {"Carbon fine on fuel CO₂":<35} {fmt(c_cfine(CGAS,ets)):>12} {fmt(c_cfine(CELEC,ets)):>12}')
print('─'*62)
s = c_save(ets); pb = c_pb(ets); n = c_npv(ets)
print(f'  Annual saving               : {fmt(s)}')
print(f'  CAPEX                       : {fmt(c_capex())}')
print(f'  Payback                     : {f"{pb:.1f} years ✅" if pb else "❌"}')
print(f'  NPV (15yr)                  : {fmt(n)}  {"✅ STRONG INVEST" if n>0 else "❌"}')
co2_saved = c_fco2(CGAS) - c_fco2(CELEC)
pct = co2_saved / (CP["cement_output"] * CO2S["total"]) * 100
print(f'\n  Fuel CO₂ saved/year         : {co2_saved:,.0f} tonnes ({pct:.1f}% of total plant CO₂)')
print(f'  Remaining {100-pct:.1f}% (calcination) requires CCS')
print(f'  Equiv. cars off road        : {co2_saved/1.2:,.0f} cars/year')
print('═'*62)

══════════════════════════════════════════════════════════════
  CEMENT TEA — BEST CASE (PPA €0.07/kWh, ETS €126/t 2030)
══════════════════════════════════════════════════════════════
                                          GAS     ELECTRIC
────────────────────────────────────────────────────────────────
  Energy cost/year                     €59.66M      €56.71M
  Fuel CO₂ (tonnes)                    267,805       76,968
  Carbon fine on fuel CO₂              €33.74M       €9.70M
────────────────────────────────────────────────────────────────
  Annual saving               : €26.99M
  CAPEX                       : €36.67M
  Payback                     : 1.4 years ✅
  NPV (15yr)                  : €222.33M  ✅ STRONG INVEST

  Fuel CO₂ saved/year         : 190,837 tonnes (29.4% of total plant CO₂)
  Remaining 70.6% (calcination) requires CCS
  Equiv. cars off road        : 159,031 cars/year
══════════════════════════════════════════════════════════════


---
## 📊 Summary

| Industry | Current Case | Best Case Payback | Best Case NPV |
|----------|-------------|-------------------|---------------|
| 🔩 Steel | ❌ Not viable | ✅ **6.8 years** | ✅ **€16M** |
| 🏗️ Cement | ❌ Not viable | ✅ **1.4 years** | ✅ **€222M** |

**The 3 levers that flip both models:**
1. 🌬️ **Renewable PPA** — electricity €0.12 → €0.07/kWh
2. 📈 **EU ETS 2030** — carbon €70 → €126/tonne (BNEF/S&P consensus)
3. 💚 **Green product premium** — BMW/Volvo/HeidelbergMaterials already paying

> *Built for HyperHeat GmbH (Cologne) Business Analyst application — Sridhar Iyer, Constructor University Bremen*